# run_simulation.ipynb

Main simulation runner. Run each section in order to build up the simulation step by step.

**Sections:**
1. Setup — install packages, connect API clients
2. Configure — choose agents, scenario, ablation flags
3. Preview — inspect agents and event timeline before running
4. Initialise simulation — load agents and seed memories
5. Run — step through tick by tick (one cell per day) or run all at once
6. Inspect — compare agents, dig into memories, read decisions
7. Export — save results to CSV / Google Sheets for analysis

**To run a different experiment variant**, change the flags in Section 2 and re-run from Section 4.

---
## Section 1 — Setup

Install packages and initialise API clients. Run once per Colab session.

In [ ]:
# Install dependencies (Colab only — comment out if running locally)
# !pip install anthropic sentence-transformers pyyaml openai --quiet

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# ── Set this to your repo path in Google Drive ────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION'

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f'Working directory: {os.getcwd()}')

In [ ]:
from src.llm.client import init_clients, init_openrouter_client, Config

# Anthropic client — required for Claude models
client_anthropic = init_clients()

# OpenRouter client — only needed if you're using GPT-4o / DeepSeek / etc.
# Uncomment when running Experiment 2 model comparison:
# client_openrouter = init_openrouter_client()
client_openrouter = None

---
## Section 2 — Configure

Set the agents, scenario, and experiment flags here. This is the only cell you need
to change between experiment variants.

**Experiment 1 ablation variants:**
- Variant 1 (full):          `use_memory=True,  use_reflection=True`
- Variant 2 (no reflection): `use_memory=True,  use_reflection=False`
- Variant 3 (no memory):     `use_memory=False, use_reflection=False`

**Experiment 2 model variants:**
- Variant 1 (mixed):  `DECISION_MODEL='claude-haiku-4-5-20251001', REFLECTION_MODEL='claude-sonnet-4-6'`
- Variant 2 (small):  both set to `'claude-haiku-4-5-20251001'`
- Variant 3 (large):  both set to `'claude-sonnet-4-6'`

In [ ]:
from src.engine.simulation import SimulationConfig
from src.llm.client import Config
from src.agents.retrieval import RetrievalConfig
from src.agents.reflection import ReflectionConfig

# ── Agents to load ────────────────────────────────────────────────────────────
# Each path points to an agent YAML with id, display_name, seed_narrative, memory_seeds.
AGENT_PATHS = [
    'config/agents/beth.yaml',        # Margaret — pragmatic complier, high frustration with system
    'config/agents/eleanor_v2.yaml',  # Eleanor  — pragmatic, institutional frustration
]

# ── Scenario ──────────────────────────────────────────────────────────────────
SCENARIO_PATH = 'config/scenarios/baseline.yaml'

# ── LLM models ────────────────────────────────────────────────────────────────
# Experiment 2 Variant 1 (mixed) is the default.
# Swap both to the same model for Variants 2 and 3.
llm_config = Config(
    DECISION_MODEL   = 'claude-sonnet-4-6',   # routine agent decisions
    REFLECTION_MODEL = 'claude-sonnet-4-6',   # belief synthesis (use larger model for quality)
    JUDGE_MODEL      = 'claude-sonnet-4-6',   # evaluation judge
)

# ── Retrieval + reflection hyperparameters ────────────────────────────────────
# Update these once locked from stage2_validation.ipynb runs.
retrieval_config = RetrievalConfig(
    top_k=12,
    recency_weight=1.0,
    importance_weight=1.0,
    relevance_weight=2.0,
    recency_decay=0.99,
    retrieval_mode='dense',
)

reflection_config = ReflectionConfig(
    threshold=100.0,
    num_questions=3,
)

# ── Ablation flags (Experiment 1) ────────────────────────────────────────────
USE_MEMORY     = True   # False → Variant 3
USE_REFLECTION = True   # False → Variant 2

# ── Assemble config ───────────────────────────────────────────────────────────
sim_config = SimulationConfig(
    scenario_path     = SCENARIO_PATH,
    agent_yaml_paths  = AGENT_PATHS,
    llm_config        = llm_config,
    retrieval_config  = retrieval_config,
    reflection_config = reflection_config,
    use_memory        = USE_MEMORY,
    use_reflection    = USE_REFLECTION,
    output_dir        = 'outputs/runs',
)

print(sim_config.describe())

---
## Section 3 — Preview

Inspect agents and the event timeline **before** running anything. No LLM calls here.
Use this to sanity-check that the right agents and events are loaded.

In [ ]:
# Preview the event timeline from the scenario YAML
import yaml
import pandas as pd

with open(SCENARIO_PATH) as f:
    scenario = yaml.safe_load(f)

print(f"Scenario: {scenario['scenario_name']}")
print(f"Duration: {scenario['duration_days']} days")
print(f"Description: {scenario['description']}\n")

events_df = pd.DataFrame([
    {
        'Day': e['day'],
        'Type': e['type'],
        'Channel': e['channel'],
        'Target': str(e.get('target_agents', 'all')),
        'Content (preview)': str(e['content'])[:80] + '...',
    }
    for e in scenario['events']
])
print(events_df.to_string(index=False))

In [ ]:
# Preview agent profiles from the YAML (no LLM calls)

for path in AGENT_PATHS:
    with open(path) as f:
        raw = yaml.safe_load(f)
    data = raw['agents'][0] if 'agents' in raw else raw

    print(f"\n{'='*60}")
    print(f"  {data['display_name']}  (id: {data['id']})")
    print(f"  Compliance: {data.get('compliance_status', 'unknown')}  "
          f"| Trust: {data.get('institutional_trust', 'unknown')}")
    print(f"  Memory seeds: {len(data.get('memory_seeds', []))}")
    print(f"  Seed narrative (first 200 chars):")
    print(f"    {data['seed_narrative'][:200]}...")
    if data.get('key_concerns'):
        print(f"  Key concerns:")
        for c in data['key_concerns'][:3]:
            print(f"    • {c}")
    print(f"{'='*60}")

---
## Section 4 — Initialise simulation

Creates the Simulation object. This **does** make LLM calls — one embedding per
memory seed per agent. Margaret has ~45 seeds, Eleanor has ~10, so expect ~55
embedding calls (fast, local model) and zero API calls unless any seeds are
missing importance scores.

Watch the output — you'll see each agent's memory stream being seeded.

In [ ]:
from src.engine.simulation import Simulation

sim = Simulation(
    sim_config        = sim_config,
    client_anthropic  = client_anthropic,
    client_openrouter = client_openrouter,
)

print('\nSimulation ready. Run Section 5 to start stepping through days.')

---
## Section 5 — Run

### Option A: Full run (all days at once)
Use this when you want to run the whole simulation and inspect afterwards.

### Option B: Step through day by day
Use this to watch the simulation unfold and inspect each tick interactively.
Re-run the step cell to advance one day at a time.

In [ ]:
# ── Option A: Full run ────────────────────────────────────────────────────────
# Uncomment to run the entire simulation at once.
# Results are in sim.tick_results and written to outputs/runs/<run_id>.jsonl

# results = sim.run(verbose=True)
# print(f'Done. {len(results)} ticks completed.')

In [ ]:
# ── Option B: Step through one day at a time ─────────────────────────────────
# Re-run this cell to advance one day. It shows what happened on the current day
# and skips silently over days with no scheduled events.

tick_result = sim.step()

if tick_result is None:
    print('Simulation complete. Run Section 6 to inspect results.')
else:
    tick = tick_result['tick']
    events = tick_result['events_processed']

    if not events:
        print(f'Day {tick}: no events scheduled.')
    else:
        for er in events:
            print(f'\n{"="*60}')
            print(f'  Day {tick} | {er["event_type"].upper()} | {er["channel"]}')
            print(f'  Targets: {er["target_display_names"]}')
            print(f'  Intervention: {er["content"][:120]}...')
            for resp in er['agent_responses']:
                print(f'\n  [{resp["agent_display_name"]}]')
                print(f'    Memories retrieved: {len(resp["retrieved_memories"])}')
                print(f'    Decision:  {resp["decision"][:110]}')
                print(f'    Reasoning: {resp["reasoning"][:110]}')
                if resp['new_reflections']:
                    print(f'    ★ REFLECTION fired — {len(resp["new_reflections"])} new insight(s)')
                    for r in resp['new_reflections']:
                        print(f'      → {r.description[:100]}')
        print(f'\n  Agent states after day {tick}:')
        for state in tick_result['agent_states']:
            print(f'    {state["display_name"]}: '
                  f'{state["memory_count"]} memories, '
                  f'{state["actions_taken_count"]} actions taken')

---
## Section 6 — Inspect results

Helper functions for exploring the simulation results. Run these at any point
during or after the simulation.

In [ ]:
# ── inspect_agent: full state for one agent ───────────────────────────────────

def inspect_agent(display_name: str):
    """Print the current state of one agent — memory count, compliance, recent actions."""
    agent = sim.agents.get(display_name)
    if agent is None:
        print(f"Agent '{display_name}' not found. Available: {list(sim.agents.keys())}")
        return
    agent.pretty_print()

# Example: inspect_agent('Margaret')

In [ ]:
# ── inspect_memory: full memory stream for one agent ─────────────────────────

def inspect_memory(display_name: str, n: int = None, memory_type: str = None):
    """
    Print the memory stream for one agent.
    
    Args:
        display_name: Agent to inspect.
        n:            Only show the n most recent memories (None = all).
        memory_type:  Filter to a type: 'observation', 'decision', 'reflection', 'conversation'.
    """
    agent = sim.agents.get(display_name)
    if agent is None:
        print(f"Agent '{display_name}' not found.")
        return
    memories = agent.memory.get_all()
    if memory_type:
        memories = [m for m in memories if m.type == memory_type]
    if n:
        memories = memories[-n:]
    print(f'\nMemory stream: {display_name} ({len(memories)} shown)')
    print('='*60)
    for m in memories:
        print(f'[#{m.id}] Day {m.timestamp} | {m.type.upper()} | importance {m.importance}/10')
        print(f'  {m.description}')
        print()

# Examples:
# inspect_memory('Margaret')
# inspect_memory('Margaret', n=10)            # 10 most recent
# inspect_memory('Eleanor', memory_type='reflection')  # reflections only

In [ ]:
# ── compare_agents: side-by-side decisions for the same event ─────────────────

def compare_agents(event_type: str = None):
    """
    Show all agents' decisions side by side, optionally filtered by event type.
    
    This is the primary tool for checking whether agents diverge as expected —
    Margaret and Eleanor should respond differently to the same intervention.
    """
    all_responses = []
    for tick_result in sim.tick_results:
        for er in tick_result['events_processed']:
            if event_type and er['event_type'] != event_type:
                continue
            for resp in er['agent_responses']:
                all_responses.append({
                    'day': tick_result['tick'],
                    'event_type': er['event_type'],
                    'channel': er['channel'],
                    'agent': resp['agent_display_name'],
                    'decision': resp['decision'],
                    'reasoning': resp['reasoning'],
                    'memories_used': len(resp['retrieved_memories']),
                    'reflections_fired': len(resp['new_reflections']),
                })

    if not all_responses:
        print('No responses yet. Run some simulation steps first.')
        return

    for r in all_responses:
        print(f'\n{"─"*60}')
        print(f'  Day {r["day"]} | {r["event_type"]} | {r["agent"]} '
              f'| {r["memories_used"]} memories used')
        print(f'  DECISION:  {r["decision"][:120]}')
        print(f'  REASONING: {r["reasoning"][:120]}')
        if r['reflections_fired']:
            print(f'  ★ Reflection fired')

# Examples:
# compare_agents()                              # all events
# compare_agents('insurance_non_renewal')       # one event type

In [ ]:
# ── show_tick_summary: all activity on one specific day ───────────────────────

def show_tick_summary(day: int):
    """Show everything that happened on a specific simulation day."""
    matches = [r for r in sim.tick_results if r['tick'] == day]
    if not matches:
        print(f'No results for day {day}. Simulation may not have reached that day yet.')
        return
    tick_result = matches[0]
    if not tick_result['events_processed']:
        print(f'Day {day}: no events were scheduled.')
        return
    for er in tick_result['events_processed']:
        print(f'\nEvent: {er["event_type"]} | Channel: {er["channel"]}')
        print(f'Content: {er["content"]}')
        for resp in er['agent_responses']:
            print(f'\n  [{resp["agent_display_name"]}]')
            print(f'  Retrieved memories ({len(resp["retrieved_memories"])}):')
            for m in resp['retrieved_memories']:
                print(f'    [{m["type"]} | imp {m["importance"]}] {m["description"][:80]}')
            print(f'  Decision:  {resp["decision"]}')
            print(f'  Reasoning: {resp["reasoning"]}')
            if resp['new_reflections']:
                print(f'  Reflections fired:')
                for r in resp['new_reflections']:
                    print(f'    → {r.description}')

# Example: show_tick_summary(42)   # see what happened on the insurance non-renewal day

In [ ]:
# ── show_all_reflections: every reflection that fired during the simulation ────

def show_all_reflections():
    """Show every reflection insight generated across all agents and ticks."""
    found = False
    for tick_result in sim.tick_results:
        for er in tick_result['events_processed']:
            for resp in er['agent_responses']:
                if resp['new_reflections']:
                    found = True
                    print(f'\nDay {tick_result["tick"]} | {resp["agent_display_name"]} '
                          f'| after {er["event_type"]}')
                    for r in resp['new_reflections']:
                        print(f'  [importance {r.importance}/10] {r.description}')
    if not found:
        print('No reflections have fired yet.')
        print('Reflections fire when cumulative memory importance exceeds the threshold '
              f'({sim.sim_config.reflection_config.threshold}).')
        total = sum(a.memory.get_cumulative_importance() for a in sim.agents.values())
        print(f'Current cumulative importance across all agents: {total}')

---
## Section 7 — Export results

After the simulation completes, save all decisions to CSV for analysis
and optionally push to Google Sheets.

In [ ]:
# Close the logger if you used step() rather than run()
# (safe to call even if already closed)
try:
    sim.close()
except Exception:
    pass

print(f'JSONL log: outputs/runs/{sim.sim_config.run_id}.jsonl')

In [ ]:
# Export all decisions to a flat CSV for analysis
import pandas as pd
import json

log_path = f'outputs/runs/{sim.sim_config.run_id}.jsonl'
decisions = []

with open(log_path) as f:
    for line in f:
        entry = json.loads(line)
        if entry['entry_type'] == 'decision':
            decisions.append({
                'tick':                  entry['tick'],
                'agent':                 entry['agent_display_name'],
                'event_type':            entry['event_type'],
                'channel':               entry['channel'],
                'decision':              entry['decision'],
                'reasoning':             entry['reasoning'],
                'memories_retrieved':    len(entry['retrieved_memories']),
            })

df = pd.DataFrame(decisions)
csv_path = f'outputs/runs/{sim.sim_config.run_id}_decisions.csv'
df.to_csv(csv_path, index=False)

print(f'{len(df)} decisions saved to {csv_path}')
display(df)

In [ ]:
# Optional: score decisions with LLM-as-a-judge
# Uncomment and run after the simulation to get judge scores for each decision.
# Warning: this makes one API call per decision row — check df.shape first.

# from src.llm.client import judge_simulation
# 
# scores = []
# with open(log_path) as f:
#     for line in f:
#         entry = json.loads(line)
#         if entry['entry_type'] != 'decision':
#             continue
#         result = judge_simulation(
#             client_anthropic   = client_anthropic,
#             config             = llm_config,
#             seed_personality   = entry['seed_personality'],
#             intervention       = entry['intervention'],
#             decision           = entry['decision'],
#             reasoning          = entry['reasoning'],
#             client_openrouter  = client_openrouter,
#         )
#         scores.append({
#             'tick':                          entry['tick'],
#             'agent':                         entry['agent_display_name'],
#             'event_type':                    entry['event_type'],
#             'behavioral_plausibility':       result.get('behavioral_plausibility'),
#             'persona_consistency':           result.get('persona_consistency'),
#             'intervention_responsiveness':   result.get('intervention_responsiveness'),
#             'critique':                      result.get('critique'),
#         })
#         print(f"  Scored day {entry['tick']} {entry['agent_display_name']}: "
#               f"BP={result.get('behavioral_plausibility')} "
#               f"PC={result.get('persona_consistency')} "
#               f"IR={result.get('intervention_responsiveness')}")
# 
# scores_df = pd.DataFrame(scores)
# scores_path = f'outputs/eval/{sim.sim_config.run_id}_scores.csv'
# scores_df.to_csv(scores_path, index=False)
# print(f'\nScores saved to {scores_path}')
# display(scores_df.groupby('agent')[['behavioral_plausibility','persona_consistency','intervention_responsiveness']].mean())